**DATA DESCRIPTION**

1. **specialty**: The specialty of the appointment
2. **appointment_time**: The time of the appointment .
3. **gender**: The gender of the patient(F,M)
4. **appointment_date**: The date of the appointment.
5. **no_show**: Indicates whether the patient showed up for the appointment.
6. **no_show_reason**: The reason for not showing up for the appointment.
7. **disability**: Indicates whether the patient has a disability.
8. **date_of_birth**: The date of birth of the patient.
9. **entry_service_date**: The date when the patient entered the service.
10. **city**: The city where the appointment took place.
11. **icd**: The International Classification of Diseases code associated with the appointment
12. **appointment_month**: The month of the appointment
13. **appointment_year**: The year of the appointment
14. **appointment_shift**: The shift of the appointment
15. **age**: The age of the patient
16. **under_12_years_old**: Indicates whether the patient is under 12 years old.
17. **over_60_years_old**: Indicates whether the patient is over 60 years old
18. **patient_needs_companion**: Indicates whether the patient needs a companion
19. **average_temp_day**: The average temperature of the day
20. **average_rain_day**: The average rainfall of the day
21. **max_temp_day**: The maximum temperature of the day
22. **max_rain_day**: The maximum rainfall of the day
23. **rainy_day_before**: Indicates whether it rained on the day before the appointment
24. **storm_day_before**: Indicates whether there was a storm on the day before the appointment
25. **rain_intensity**: The intensity of rain on the day of the appointment
26. **heat_intensity**: The intensity of heat, but the description is incomplete or unclear.



# **Loading Python libraries**

In [ ]:
!pip install ipython-autotime
%load_ext autotime

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

In [ ]:
import random
random.seed(42)
np.random.seed(42)

In [ ]:
!python --version
!pip list 2>/dev/null | grep -i -E "scikit-learn|pandas|numpy|tensorflow|imbalanced"

# ***Dataset***

In [ ]:
df = pd.read_csv("/content/Medical-appointments-no-show-en.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

# ***Data Cleaning***

# **Check for Null values in the dataset**

In [ ]:
df.isna().sum()

In [ ]:
len(df)

In [ ]:
df = df.drop_duplicates()

In [ ]:
len(df)

In [ ]:
# Select the categorical columns for encoding
categorical_columns = ['specialty', 'appointment_time', 'gender', 'no_show_reason', 'disability', 'city', 'icd', 'appointment_month', 'appointment_shift']

# Perform one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_columns)

# Display the encoded DataFrame
print(df_encoded.head())

In [ ]:
# Calculate correlation
correlation = df.corr(numeric_only=True)

# Display the correlation matrix
print(correlation)

In [ ]:
# Visualize missing values
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values")
plt.show()

In [ ]:
# Specify columns to drop
columns_to_drop = ['no_show_reason', 'disability', 'icd', 'city']

# Filter out columns that exist in the dataset
columns_to_drop_existing = [col for col in columns_to_drop if col in df.columns]

# Drop the existing columns
df = df.drop(columns_to_drop_existing, axis=1)

# Print the updated DataFrame
print(df)

# **HANDLING MISSING VALUE IN SPECIALITIES**

In [ ]:
import pandas as pd

# Define important specialties and default specialty
important_specialties = ['physiotherapy', 'occupational therapy', 'speech therapy', 'assist','enf' ,'pedagogo' ]

default_specialty = 'General Medicine'  # Choose a default specialty for cases where patient_needs_companion is 0

# Replace missing values in 'specialty' based on the value of 'patient_needs_companion'
df['specialty'] = df.apply(lambda row: important_specialties[0] if row['patient_needs_companion'] == 1 and pd.isna(row['specialty']) else default_specialty if row['patient_needs_companion'] == 0 and pd.isna(row['specialty']) else row['specialty'], axis=1)

# Check if there are any missing values left in the 'specialty' column
print("Missing values in 'specialty' after handling:")
print(df['specialty'].isna().sum())


In [ ]:
df.loc[:, ['date_of_birth', 'age']]

In [ ]:
df['date_of_birth'].head(10)

HANDLING MISSING VALUE IN AGE

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Create a new column 'age_category'
def get_age_category(row):
    if row['under_12_years_old'] == 0 and row['over_60_years_old'] == 0:
        return 'Between'
    else:
        return 'Not between'

df['age_category'] = df.apply(get_age_category, axis=1)

# Encode 'age_category', 'under_12_years_old', and 'over_60_years_old' into a single column
encoder = LabelEncoder()

# Encode age_category column
df['age_category_encoded'] = encoder.fit_transform(df['age_category'])

# Encode under_12_years_old column
df['under_12_years_old_encoded'] = encoder.fit_transform(df['under_12_years_old'])

# Encode over_60_years_old column
df['over_60_years_old_encoded'] = encoder.fit_transform(df['over_60_years_old'])

# Combine encoded columns into a single column
df['age_group'] = df['under_12_years_old_encoded'] + df['over_60_years_old_encoded'] + df['age_category_encoded']

# Drop redundant columns
df.drop(['under_12_years_old', 'over_60_years_old', 'age_category', 'age_category_encoded', 'under_12_years_old_encoded', 'over_60_years_old_encoded'], axis=1, inplace=True)




In [ ]:
df

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression



# Split the dataset into two sets: one with non-missing 'age' values and one with missing 'age' values (NaNs)
df_age_present = df.dropna(subset=['age'])
df_age_missing = df[df['age'].isna()]

# Create feature matrix (X) and target vector (y) for the training set
X_train = df_age_present[['age_group']]  # Feature: age_category
y_train = df_age_present['age']             # Target: age

# Create feature matrix (X) for the set with missing 'age' values
X_test = df_age_missing[['age_group']]

# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict the missing 'age' values
predicted_age = model.predict(X_test)

# Fill the missing 'age' values in the original dataset with the predicted values
df.loc[df['age'].isna(), 'age'] = predicted_age



In [ ]:
import pandas as pd
# Drop the 'date_of_birth' column
df.drop(columns=['date_of_birth'], inplace=True)


In [ ]:
assert df['age'].isna().sum() == 0, "age still has missing values"
assert 'date_of_birth' not in df.columns
assert set(df['no_show'].unique()) <= {'yes', 'no'}

In [ ]:
df['age'].isna().sum() / len(df)

In [ ]:
df['age'].unique()

In [ ]:
df['age'].isna().sum()

In [ ]:
df['age'].head()

In [ ]:
df['age'].value_counts()

In [ ]:
# Create a histogram using Seaborn
sns.histplot(df['age'], bins=5, kde=True)  # You can adjust the number of bins as needed

# Adding labels and title
plt.xlabel("X-axis Label")
plt.ylabel("Frequency")
plt.title("Histogram Example")

# Display the plot
plt.show()

# ***Extracting date from Scheduled Day***

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1) Convert entry_service_date to datetime (day/month/year).
#    Malformed strings such as "29/05/201" become NaT.
df['entry_service_date'] = pd.to_datetime(df['entry_service_date'], errors='coerce', dayfirst=True)

# 2) Replace missing/malformed dates with the earliest date in the dataset
default_date = df['entry_service_date'].min()
df['entry_service_date'] = df['entry_service_date'].fillna(default_date)

# 3) Handle remaining missing values using forward fill
df.ffill(inplace=True)

# 4) Feature Engineering
df['appointment_date'] = pd.to_datetime(df['appointment_date'], dayfirst=True)
df['days_between'] = (df['appointment_date'] - df['entry_service_date']).dt.days
# Add new column for scheduling date
df['scheduling_date'] = df['appointment_date'] - pd.to_timedelta(df['days_between'], unit='d')

# 5) Encode categorical variables
label_encoder = LabelEncoder()
df['appointment_month'] = label_encoder.fit_transform(df['appointment_month'])
df['rain_intensity'] = label_encoder.fit_transform(df['rain_intensity'])

# 6) Select features and target variable
X = df[['days_between', 'rain_intensity']]
y = df['no_show']

# 7) Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 8) Initialize and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 9) Make predictions
y_pred = model.predict(X_test)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# Handle missing values using forward fill
df.ffill(inplace=True)

# Feature Engineering (dates are day/month/year)
df['appointment_date'] = pd.to_datetime(df['appointment_date'], dayfirst=True)
df['entry_service_date'] = pd.to_datetime(df['entry_service_date'], dayfirst=True)
df['days_between'] = (df['appointment_date'] - df['entry_service_date']).dt.days
# Add new column for scheduling date
df['scheduling_date'] = df['appointment_date'] - pd.to_timedelta(df['days_between'], unit='d')

# Encode categorical variables
label_encoder = LabelEncoder()
df['appointment_month'] = label_encoder.fit_transform(df['appointment_month'])
df['rain_intensity'] = label_encoder.fit_transform(df['rain_intensity'])

# Select features and target variable
X = df[[ 'days_between', 'rain_intensity']]
y = df['no_show']

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

In [ ]:
df

In [ ]:
len(df[df["appointment_date"] < df["scheduling_date"]])


'AppointmentDay' has no value for time of the day and it has many values(38568) smaller than 'ScheduledDay' which is not possible !
The reason for this problem is that probably these appointments happened at the same day that they're booked but because we don't have the exact hour for 'AppointmentDay' their difference is negative.
To solve this problem, we added 23 hrs and 59 min and 59 secs to the 'AppointmentDay'. Now all 'AppointmentDay' are still at the same the same day

In [ ]:
df['appointment_date'] = df['scheduling_date'] + pd.Timedelta('1d') - pd.Timedelta('1s')
df.head()

In [ ]:
len(df[df["appointment_date"] < df["scheduling_date"]])


In [ ]:
def parse_date(date):
    if date == '':
        return None
    else:
        return (dt.strptime(date, "%Y-%m-%dT%H:%M:%S%fz"))

def parse_maybe_int(i):
    if i == '':
        return None
    else:
        return int(i)

def get_hour(date):
    if date == '':
        return None
    else:
        return parse_maybe_int(str(parse_date(date)).split(' ')[1].split(':')[0])

def get_day(date):
    if date == '':
        return None
    else:
        return parse_date(date).strftime('%A')

def get_month(date):
    if date == '':
        return None
    else:
        return parse_maybe_int(str(parse_date(date)).split('-')[1])

def get_year(date):
    if date == '':
        return None
    else:
        return parse_maybe_int(str(parse_date(date)).split('-')[0])

In [ ]:
# Extract hour from the 'HH:MM' format
df['scheduledhour'] = pd.to_datetime(df['appointment_time'], format='%H:%M').dt.hour

# Assuming 'appointment_date' is in the format '31/08/2021'
df['scheduleddayofweek'] = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y').dt.day_name()
df['scheduledmonth'] = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y').dt.month
df['scheduledyear'] = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y').dt.year

# Assuming 'appointment_date' is in the format '31/08/2021'
df['appointmentdayofweek'] = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y').dt.day_name()
df['appointmentmonth'] = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y').dt.month
df['appointmentyear'] = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y').dt.year

In [ ]:
df.drop(['appointment_month','appointment_time'], axis=1, inplace=True)

In [ ]:
df

In [ ]:
df.head()

In [ ]:
df.isna().sum()

# ***Adding "Lead days" column***

In [ ]:
df['LeadDays'] = (df["appointment_date"] - df["scheduling_date"]).dt.days


In [ ]:
df.head()


# ***We classify Lead Days into 6 categories
***

In [ ]:
def lead_day_category(LeadDay):
    if(LeadDay<=2):
        return 'Within 2 Days'
    elif(LeadDay>2 and LeadDay<=7):
        return 'Within 1 Week'
    elif(LeadDay>7 and LeadDay<=14):
        return 'Within 2 Weeks'
    elif(LeadDay>14 and LeadDay<=21):
        return 'Within 3 Weeks'
    elif(LeadDay>21 and LeadDay<=28):
        return 'Within 4 Weeks'
    elif(LeadDay>28):
        return 'After 1 Month'

In [ ]:
df

In [ ]:
# Handle missing values for categorical columns
categorical_cols = [ 'rain_intensity', 'heat_intensity', 'gender','entry_service_date']
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# Handle missing values for numeric columns
numeric_cols = [ 'average_temp_day', 'average_rain_day', 'max_temp_day', 'max_rain_day',
                'rainy_day_before', 'storm_day_before']
imputer = SimpleImputer(strategy='mean')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
# Drop the letter 'i' from the values in the 'gender' column
###################################################################################################
df['gender'] = df['gender'].str.replace('I', 'F')






# ***Converting Categorical values to Numeric values
***

In [ ]:
# Map 'yes' to 1 and 'no' to 0 in the 'no_show' column
df['no_show'] = df['no_show'].map({'yes': 1, 'no': 0})
# Replace 'yes' with 1 and 'no' with 0 in the 'no_show' column
df['no_show'] = df['no_show'].replace({'yes': 1, 'no': 0})

In [ ]:
df['gender'] = df['gender'].map({'M': 0, 'F': 1})
df['appointment_shift'] = df['appointment_shift'].map({'morning': 0, 'afternoon': 1, 'evening': 2})

In [ ]:
df.head()

In [ ]:
df.isnull().sum().any()

In [ ]:
# Visualize missing values
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values")
plt.show()

In [ ]:
# Calculate correlation
correlation = df.corr(numeric_only=True)

# Display the correlation matrix
print(correlation)

In [ ]:
# List of variables for which histograms will be created
variables = [
    "specialty", "scheduledhour", "gender", "no_show",
    "appointment_year", "appointment_shift", "age",
    "age_group", "patient_needs_companion", "average_temp_day",
    "average_rain_day", "max_temp_day", "max_rain_day", "rainy_day_before",
    "storm_day_before", "rain_intensity", "heat_intensity",
    "appointmentdayofweek", "appointmentmonth", "appointmentyear"
]

# Plot histograms for each variable
for variable in variables:
    plt.figure(figsize=(16,16));
    df[variable].hist()
    plt.xlabel(variable.capitalize())
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {variable.capitalize()}")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming you have a DataFrame named 'df' and a list of variables 'variables'
# Replace this with your actual DataFrame and variables list

# Set the number of columns and rows for the subplot grid
num_cols = 3
num_rows = len(variables) // num_cols + (len(variables) % num_cols > 0)

# Create subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(18, 5 * num_rows))

# Flatten the axes array for easier indexing
axes = axes.flatten()

# Plot histograms for each variable in the grid
for i, variable in enumerate(variables):
    ax = axes[i]
    df[variable].hist(ax=ax)
    ax.set_xlabel(variable.capitalize())
    ax.set_ylabel("Frequency")
    ax.set_title(f"Distribution of {variable.capitalize()}")

# Remove empty subplots
for i in range(len(variables), num_cols * num_rows):
    fig.delaxes(axes[i])

# Adjust layout
plt.tight_layout()
plt.show()


In [ ]:
# Count the number of appointments for each specialty
specialty_counts = df['specialty'].value_counts()

# Create a bar chart
plt.figure(figsize=(16, 6))
plt.bar(specialty_counts.index, specialty_counts.values)
plt.xlabel('Specialty')
plt.ylabel('Number of Appointments')
plt.title('Number of Appointments by Specialty')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Count the number of appointments for each no-show status
noshow_counts = df['no_show'].value_counts()

# Create a pie chart
plt.figure(figsize=(6, 6))
plt.pie(noshow_counts.values, labels=['No Show', 'Show'], autopct='%1.1f%%', startangle=90, colors=['blue', 'yellow'])
plt.title('No-Show Status')
plt.axis('equal')
plt.show()


In [ ]:
import seaborn as sns

# Set the style for the plot
sns.set(style="whitegrid")

# Create a figure with subplots
fig, ax = plt.subplots(figsize=(10, 6))

# Plot histogram
sns.histplot(df['age'], bins=20, kde=False, color='skyblue', edgecolor='k', ax=ax)

# Plot KDE
sns.kdeplot(df['age'], color='red', ax=ax)

# Set labels and title
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.title('Distribution of Patient Ages')

# Show the plot
plt.show()

In [ ]:
# Count the number of appointments for each combination of gender and no-show status
gender_noshow_counts = df.groupby(['gender', 'no_show']).size().unstack()

# Create a stacked bar chart with blue and orange colors
plt.figure(figsize=(8, 6))
ax = gender_noshow_counts.plot(kind='bar', stacked=True, color=['blue', 'orange'])
plt.xlabel('Gender')
plt.ylabel('Number of Appointments')
plt.title('No-Show Status by Gender')
plt.xticks([0, 1], labels=['Male', 'Female'], rotation=0)  # Set x-axis labels

# Set legend labels
plt.legend( labels=['No Show', ' Show'], loc='upper right')

# Show the plot
plt.show()


In [ ]:
# Count the number of appointments for each combination of no-show status and specialty
noshow_specialty_counts = df.groupby(['no_show', 'specialty']).size().unstack()

# Create a stacked bar chart
plt.figure(figsize=(10, 6))
ax = noshow_specialty_counts.plot(kind='bar', stacked=True)
plt.xlabel('No-Show Status')
plt.ylabel('Number of Appointments')
plt.title('No-Show Status by Specialty')
plt.legend(title='Specialty')
plt.xticks([0, 1], labels=['No Show', 'Show'], rotation=0)  # Set x-axis labels

# Show the plot
plt.show()

In [ ]:
showed_up = df.no_show == True
no_showed = df.no_show == False

In [ ]:
df['no_show'].unique()

In [ ]:
# Count the number of appointments for each no-show status
no_show_counts = df['no_show'].value_counts().rename({0: 'No Show', 1: 'Show'})

# Display the counts with updated labels
print(no_show_counts)


In [ ]:
# Calculate the percentage for each no-show status
showed_percentage = df['no_show'].value_counts(normalize=True).rename({0: 'No Show', 1: 'Show'}) * 100

# Display the percentages with updated labels
print(showed_percentage)


In [ ]:
df['gender'].unique() # M': 0, 'F': 1

In [ ]:

# Assuming 'gender' column values are 0 for Male and 1 for Female
# If it's the other way around, you can reverse the order in xticklabels

# Create a count plot
sns.countplot(x='gender', data=df)

# Set x-axis labels
plt.xticks([0, 1], labels=['Male', 'Female'])

# Set labels and title
plt.xlabel('Gender')
plt.ylabel('Count')
plt.title('Count of Appointments by Gender')

# Show the plot
plt.show()


In [ ]:
# Count the number of appointments for each gender
gender_counts = df['gender'].value_counts()

# Rename the index labels for better clarity
gender_counts = gender_counts.rename({0: 'Male', 1: 'Female'})

# Display the counts
print(gender_counts)


In [ ]:
# Calculate the percentage of appointments for each gender
gender_percentage = df['gender'].value_counts(normalize=True) * 100

# Rename the index labels for better clarity
gender_percentage = gender_percentage.rename({0: 'Male', 1: 'Female'})

# Display the percentages
print(gender_percentage)


In [ ]:
# Group by gender and find the mean value of show rate
gender_show_rate = df.groupby('gender')['no_show'].mean().rename({0: 'Male', 1: 'Female'})

# Display the mean show rate
print(gender_show_rate)


In [ ]:
#Show up rate (gender)
df.groupby('gender')['no_show'].mean().plot(kind='bar', alpha=.7, color=['blue', 'orange']);

# title and labels
plt.xticks([0, 1], ['Female', 'Male'])
plt.ylabel('Show rate', fontsize=18)
plt.xlabel('Gender', fontsize=18)
plt.title('Appointment show rate per gender');

In [ ]:
df.gender[showed_up].hist(alpha=0.5, label = 'showed up to appointment')
df.gender[no_showed].hist(alpha=0.5, label = 'missed appointment');
plt.legend();

In [ ]:
# Assuming 'showed_up' and 'no_showed' are boolean masks for the corresponding conditions
# Replace these with your actual conditions

showed_up = df['no_show'] == 0
no_showed = df['no_show'] == 1

# Create histograms for 'showed up' and 'missed appointment'
plt.figure(figsize=(8, 6))

df.gender[showed_up].hist(alpha=0.5, label='Showed up to appointment')
df.gender[no_showed].hist(alpha=0.5, label='Missed appointment')

# Set x-axis labels
plt.xticks([0, 1], labels=['Male', 'Female'])

# Set labels and title
plt.xlabel('Gender')
plt.ylabel('Frequency')
plt.title('Distribution of Appointments by Gender and No-Show Status')

# Add legend
plt.legend()

# Show the plot
plt.show()


In [ ]:
# Filter DataFrame for no_show == 1 (yes)
no_show_yes = df[df['no_show'] == 1]

# Create histogram
plt.hist([df.age, no_show_yes.age], bins=10, label=['All', 'No Show'], alpha=0.7)
plt.title('Distribution of No Show and Age')
plt.xlabel('Age', fontsize=18)
plt.ylabel('Count', fontsize=18)
plt.legend()
plt.show()

In [ ]:
df.scheduleddayofweek.value_counts(normalize=True)

In [ ]:
# Count the occurrences of each day of the week and calculate the percentage
(df.scheduleddayofweek.value_counts(normalize=True) * 100).plot(kind='bar', alpha=0.7)

# Set title and labels for the plot
plt.title('Proportion scheduleddayofweek')  # Set the title of the plot
plt.xlabel('scheduleddayofweek', fontsize=18)  # Set the x-axis label with a font size of 18
plt.ylabel('Percentage', fontsize=18)  # Set the y-axis label with a font size of 18
plt.show()


In [ ]:
# Calculate the mean show rate for each day of the week and plot as percentages
(df.groupby('scheduleddayofweek')['no_show'].mean() * 100).plot(kind='bar', alpha=0.7)

# Set title and labels for the plot
plt.title('Appointment show rate by scheduleddayofweek')
plt.xlabel('scheduleddayofweek', fontsize=18)
plt.ylabel('Show rate (%)', fontsize=18)
plt.show()


In [ ]:
# Define a mapping of hour to time
hour_to_time = {
    7: '7:00 AM', 8: '8:00 AM', 9: '9:00 AM', 10: '10:00 AM', 11: '11:00 AM',
    12: '12:00 PM', 13: '1:00 PM', 14: '2:00 PM', 15: '3:00 PM', 16: '4:00 PM',
    17: '5:00 PM', 18: '6:00 PM'
}

# Calculate the normalized counts (proportions) for each scheduled hour
scheduled_hour_proportions = df['scheduledhour'].value_counts(normalize=True) * 100

# Display the breakdown for all scheduled hours with time
for hour, proportion in scheduled_hour_proportions.items():
    time = hour_to_time.get(hour, f'{hour}:00')
    print(f"{hour} ({time}): {proportion:.2f}% of appointments are scheduled at this time.")


In [ ]:
# Define a mapping of day of the week to name
day_to_name = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}

# Calculate the mean show rate for each day of the week
day_of_week_show_rate = df.groupby('scheduleddayofweek')['no_show'].mean() * 100

# Display the breakdown for all scheduled days of the week
for day, show_rate in day_of_week_show_rate.items():
    day_name = day_to_name.get(day)
    if day_name is not None:
        print(f"{day} ({day_name}): {show_rate:.2f}% show rate")

# Create a bar plot
day_of_week_show_rate.plot(kind='bar', alpha=0.7, figsize=(10, 6))

# Set title and labels for the plot
plt.title('Appointment Show Rate by Scheduled Day of the Week', fontsize=18)
plt.xlabel('Scheduled Day of the Week', fontsize=16)
plt.ylabel('Show Rate (%)', fontsize=16)

# Show the plot
plt.show()


In [ ]:
# Define a mapping of hour to time
hour_to_time = {
    7: '7:00 AM', 8: '8:00 AM', 9: '9:00 AM', 10: '10:00 AM', 11: '11:00 AM',
    12: '12:00 PM', 13: '1:00 PM', 14: '2:00 PM', 15: '3:00 PM', 16: '4:00 PM',
    17: '5:00 PM', 18: '6:00 PM'
}

# Calculate the normalized counts (proportions) for each scheduled hour
scheduled_hour_proportions = df['scheduledhour'].value_counts(normalize=True) * 100

# Display the breakdown for all scheduled hours with time
for hour, proportion in scheduled_hour_proportions.items():
    time = hour_to_time.get(hour, f'{hour}:00')
    print(f"{hour} ({time}): {proportion:.2f}% of appointments are scheduled at this time.")

# Create a bar plot
plt.figure(figsize=(12, 6))
scheduled_hour_proportions.sort_index().plot(kind='bar', alpha=0.7)

# Set title and labels for the plot
plt.title('Proportion of Appointments by Scheduled Hour', fontsize=18)
plt.xlabel('Scheduled Hour', fontsize=16)
plt.ylabel('Proportion (%)', fontsize=16)

# Show the plot
plt.show()


In [ ]:
# Calculate the mean show rate for each scheduled hour and plot as percentages
(df.groupby('scheduledhour')['no_show'].mean() * 100).plot(kind='bar', alpha=0.7, figsize=(10, 6))

# Set title and labels for the plot
plt.title('Appointment Show Rate by Scheduled Hour', fontsize=18)
plt.xlabel('Scheduled Hour', fontsize=16)
plt.ylabel('Show Rate (%)', fontsize=16)

# Show the plot
plt.show()


# ***Machine Leaning***

In [ ]:
# Encode the target variable 'no_show' to numerical values (0 and 1)
le = LabelEncoder()
df['no_show_encoded'] = le.fit_transform(df['no_show'])



In [ ]:
df

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import pandas as pd


# Define features and target variable
features = ['specialty', 'gender', 'appointment_year',
            'appointment_shift', 'age',
            'patient_needs_companion', 'average_temp_day', 'average_rain_day',
            'max_temp_day', 'max_rain_day', 'rainy_day_before', 'storm_day_before',
            'rain_intensity', 'heat_intensity', 'scheduledhour', 'scheduleddayofweek',
            'scheduledmonth', 'scheduledyear', 'appointmentdayofweek', 'appointmentmonth',
            'appointmentyear']
target = 'no_show_encoded'

# Features and target variable
X = df[features]  # Use all columns listed in 'features'
y = df[target]  # Use the target variable 'no_show_encoded'


# Encode categorical features
label_encoders = {}
for feature in features:
    if df[feature].dtype == 'object':  # Check if the feature is categorical
        label_encoders[feature] = LabelEncoder()
        df[feature] = label_encoders[feature].fit_transform(df[feature])

# Split the data into features (X) and target variable (y)
X = df[features]
y = df[target]

# Initialize StratifiedKFold with the desired number of folds
num_folds = 5
stratified_kfold = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

# Perform cross-validation
fold_index = 1
for train_index, test_index in stratified_kfold.split(X, y):
    train_data, test_data = X.iloc[train_index], X.iloc[test_index]
    train_labels, test_labels = y.iloc[train_index], y.iloc[test_index]

    # Process the fold
    print(f"Fold {fold_index}:")
    print("Training Data:")
    print(train_data)
    print(train_labels)
    print()
    print("Testing Data:")
    print(test_data)
    print(test_labels)
    print()

    fold_index += 1



In [ ]:
print(X.shape)
print(df['no_show_encoded'].value_counts())

# ***Random Forest ***

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Initialize the model
model = RandomForestClassifier()

# Perform cross-validation
cv_scores = []
for fold_index, (train_index, test_index) in enumerate(stratified_kfold.split(X, y), 1):
    train_data, test_data = X.iloc[train_index], X.iloc[test_index]
    train_labels, test_labels = y.iloc[train_index], y.iloc[test_index]

    # Train the model
    model.fit(train_data, train_labels)

    # Evaluate the model on the training data
    train_predictions = model.predict(train_data)
    train_accuracy = accuracy_score(train_labels, train_predictions)
    print(f"Training Accuracy (Fold {fold_index}): {train_accuracy}")

    # Evaluate the model on the testing data
    test_predictions = model.predict(test_data)
    test_accuracy = accuracy_score(test_labels, test_predictions)
    print(f"Testing Accuracy (Fold {fold_index}): {test_accuracy}")

    cv_scores.append(test_accuracy)

print(f"Mean Testing Accuracy: {sum(cv_scores) / len(cv_scores)}")  # Print the mean testing accuracy across all folds


# ***Printing the importance of each column
***

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt


# Get feature importances
feature_importances = model.feature_importances_
feature_names = X.columns

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(feature_names, feature_importances, color='skyblue')
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Feature Importance Plot for random forest')
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# X_train: Features for training
# y_train: Target variable for training
# X_test: Features for testing
# y_test: Target variable for testing


In [ ]:
from imblearn.over_sampling import RandomOverSampler

# Initialize the RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Resample the data
X_train_balanced, y_train_balanced = ros.fit_resample(X_train, y_train)


In [ ]:
y_train_balanced.value_counts()

# ***GaussianNB***

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance

# Concatenate the training and testing data
all_data_imputed = np.concatenate([X_train, X_test])
all_labels = np.concatenate([y_train, y_test])

# Initialize the Gaussian Naive Bayes model
model = GaussianNB()

# Perform cross-validation
cv_scores = cross_val_score(model, all_data_imputed, all_labels, cv=5)

# Print the cross-validation scores
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())
print("Standard Deviation of CV Scores:", cv_scores.std())

# Fit the model on the entire data
model.fit(all_data_imputed, all_labels)

# Evaluate the model on the training data
train_predictions = model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
print("\nTraining Accuracy:", train_accuracy)

# Evaluate the model on the testing data
test_predictions = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print("Testing Accuracy:", test_accuracy)

# Perform permutation importance
perm_importance = permutation_importance(model, all_data_imputed, all_labels, n_repeats=10, random_state=42)

# Get importance scores
importance_scores = perm_importance.importances_mean






# ****Printing the importance of each column


In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_names, importance_scores, color='skyblue')
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Permutation Importance Plot for GaussianNB')
plt.show()


# ***KNeighbors***

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance

# Define the parameter grid for KNN
param_grid = {
    'n_neighbors': range(1, 6),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

# Create KNN classifier
knn = KNeighborsClassifier()

# Create GridSearchCV for KNN
grid_search = GridSearchCV(estimator=knn, param_grid=param_grid, cv=5, scoring='accuracy')

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best score for KNN
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Parameters:", best_params)
print("Best Score:", best_score)

# Create KNN classifier with the best parameters
knn_best = KNeighborsClassifier(**best_params)

# Fit the KNN model with the best parameters to the training data
knn_best.fit(X_train, y_train)

# Evaluate the model on the training data
train_predictions = knn_best.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
print("Training Accuracy:", train_accuracy)

# Evaluate the model on the testing data
test_predictions = knn_best.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print("Testing Accuracy:", test_accuracy)

# Perform permutation importance
results = permutation_importance(knn_best, X_test, y_test, scoring='accuracy')



# **Printing the importance of each column**

In [ ]:
import matplotlib.pyplot as plt

# Extract feature names
feature_names = X.columns

# Get importance scores
importance_scores = results.importances_mean

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_names, importance_scores, color='skyblue')
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Permutation Importance Plot for KNN Classifier')
plt.show()


# ***DecisionTree***

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score

# Define the parameter grid for Decision Tree
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10]
}

# Create Decision Tree classifier
dt = DecisionTreeClassifier()

# Create GridSearchCV for Decision Tree
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, scoring='accuracy')

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best score for Decision Tree
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Parameters:", best_params)
print("Best Score:", best_score)

# Create Decision Tree classifier with the best parameters
dt_best = DecisionTreeClassifier(**best_params)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Fit the Decision Tree model with the best parameters and selected features to the training data
dt_best.fit(X_train, y_train)

# Evaluate the model on the training data
train_predictions = dt_best.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
print("\nTraining Accuracy:", train_accuracy)

# Evaluate the model on the testing data
test_predictions = dt_best.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print("Testing Accuracy:", test_accuracy)

# Perform cross-validation with the best model and selected features
cv_scores = cross_val_score(dt_best, X_train, y_train, cv=5)
print("\nCross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())
print("Standard Deviation of CV Scores:", cv_scores.std())


# ***print the important feature***

In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_names, feature_importances, color='skyblue')
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Feature Importance Plot for Decision Tree Classifier')
plt.show()




# ***LogisticRegression***

In [112]:
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# parameter grid for logistic regression
param_grid = {
    'penalty': ['l1', 'l2'],  # Regularization type
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  # Inverse of regularization strength
}

# Create a logistic regression model
logreg = LogisticRegression(solver='liblinear', random_state=42)

# Perform GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(estimator=logreg, param_grid=param_grid, cv=5)

# Fit the grid search to our data
grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_params_
print("Best Parameters:", best_params)

# Create a logistic regression model with the best parameters
best_logreg = LogisticRegression(solver='liblinear', random_state=42, **best_params)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Fit the model to the training data
best_logreg.fit(X_train, y_train)

# Evaluate the model on the training data
train_predictions = best_logreg.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
print("Training Accuracy:", train_accuracy)

# Evaluate the model on the testing data
test_predictions = best_logreg.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print("Testing Accuracy:", test_accuracy)

# Perform cross-validation with the best model and all data
cv_scores = cross_val_score(best_logreg, X_train, y_train, cv=5)
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())
print("Standard Deviation of CV Scores:", cv_scores.std())

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

Best Parameters: {'C': 0.001, 'penalty': 'l1'}
Training Accuracy: 0.8958658752473185
Testing Accuracy: 0.9017076218242399
Cross-Validation Scores: [0.89588756 0.89588756 0.89588756 0.89583333 0.89583333]
Mean CV Score: 0.8958658684712824
Standard Deviation of CV Scores: 2.656482889534066e-05
time: 6min 39s (started: 2026-11-18 14:33:07 +00:00)


# ***print the feature importance***

In [ ]:
import matplotlib.pyplot as plt

# Get the coefficients (feature importances)
coefficients = best_logreg.coef_[0]

# Get feature names
feature_names = X_train.columns

# Create a bar plot for feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_names, coefficients)
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Feature Importance - Logistic Regression')
plt.show()




In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred = best_logreg.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print((best_logreg.coef_ != 0).sum(), "non-zero coefficients of", best_logreg.coef_.shape[1])

bal = LogisticRegression(solver='liblinear', class_weight='balanced',
                         penalty=best_params['penalty'], C=best_params['C'], random_state=42)
bal.fit(X_train, y_train)
print(confusion_matrix(y_test, bal.predict(X_test)))
print(classification_report(y_test, bal.predict(X_test)))

# ***RandomForest & logistic regression***

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Define Models with hyperparameter grids for tuning
Models = [
    ('Logistic Regression', LogisticRegression(random_state=42), {
        'C': [0.001, 0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2']
    }),
    ('Random forest', RandomForestClassifier(random_state=42), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    })
]

# Iterate through Models
for clf_name, clf, param_grid in Models:
    print(f"Model: {clf_name}")
    grid_search = GridSearchCV(clf, param_grid, cv=5, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    # Get the best parameters and model
    best_params = grid_search.best_params_
    best_model = grid_search.best_estimator_

    # Fit the best model on the entire training data
    best_model.fit(X_train, y_train)

    # Print the best parameters
    print("Best Parameters:", best_params)

    # Print the feature importance
    if clf_name == 'Random forest':
        importance = best_model.feature_importances_
        for i, imp in enumerate(importance):
            print(f'Feature {i}: {imp}')

    # Predict on test data
    y_test_predict = best_model.predict(X_test)

    # Calculate test accuracy
    test_accuracy = accuracy_score(y_test, y_test_predict)
    print(f"Test Accuracy: {test_accuracy}\n")

    # Plot feature importance
    if clf_name == 'Random forest':
        feature_importances = best_model.feature_importances_
        plt.figure(figsize=(10, 6))
        plt.barh(X_train.columns, feature_importances)
        plt.xlabel('Feature Importance')
        plt.ylabel('Feature')
        plt.title('Feature Importance - Random Forest')
        plt.show()


# ***Multinomial Naive Bayes***

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB

# Define the parameter grid for Multinomial Naive Bayes
param_grid = {
    'alpha': [0.1, 0.5, 1.0],  # Smoothing parameter
    'fit_prior': [True, False]  # Whether to learn class prior probabilities
}

# Create Multinomial Naive Bayes classifier
nb = MultinomialNB()

# Create GridSearchCV for Multinomial Naive Bayes
grid_search = GridSearchCV(estimator=nb, param_grid=param_grid, cv=5, scoring='accuracy')

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best score for Multinomial Naive Bayes
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Parameters:", best_params)
print("Best Score:", best_score)

# Create Multinomial Naive Bayes classifier with the best parameters
nb_best = MultinomialNB(**best_params)

# Fit the Multinomial Naive Bayes model with the best parameters to the training data
nb_best.fit(X_train, y_train)

# Evaluate the model on the training data
train_predictions = nb_best.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
print("Training Accuracy:", train_accuracy)

# Evaluate the model on the testing data
test_predictions = nb_best.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print("Testing Accuracy:", test_accuracy)

# Perform cross-validation with the best model and selected features
cv_scores = cross_val_score(nb_best, X_train, y_train, cv=5)
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())
print("Standard Deviation of CV Scores:", cv_scores.std())

# Plot feature importance (if applicable)
# Multinomial Naive Bayes does not have a built-in feature importance attribute


# ***print the feature important and SVC***

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Define hyperparameters for the SVM model
svm_params = {
    'kernel': 'linear',  # Use linear kernel
    'C': 1.0,  # Regularization parameter
}

# Create the SVM model with specified hyperparameters
svm_model = SVC(**svm_params)

# Train the SVM model on the training data
svm_model.fit(X_train, y_train)

# Evaluate the model on the training data
train_predictions_svm = svm_model.predict(X_train)
train_accuracy_svm = accuracy_score(y_train, train_predictions_svm)
print("Training Accuracy (SVM):", train_accuracy_svm)

# Evaluate the model on the testing data
test_predictions_svm = svm_model.predict(X_test)
test_accuracy_svm = accuracy_score(y_test, test_predictions_svm)
print("Testing Accuracy (SVM):", test_accuracy_svm)


In [ ]:
# Get the feature importances from the SVM model
feature_importances = np.abs(svm_model.coef_[0])

# Plot the feature importances with corresponding names
plt.figure(figsize=(10, 6))
plt.bar(feature_names, feature_importances, color='skyblue')
plt.xlabel('Feature Name')
plt.ylabel('Feature Importance')
plt.title('Feature Importance for SVM')

# Set the x-axis labels to feature names
plt.xticks(range(len(feature_names)), feature_names, rotation=90)

plt.show()





# ***Neural Network***

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

# Scale the training data
scaler = StandardScaler()
train_data_scaled = scaler.fit_transform(X_train)

# Initialize StratifiedKFold for cross-validation
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Initialize lists to store accuracies
train_accuracies_nn = []
test_accuracies_nn = []

# Define hyperparameters for the neural network
learning_rate = 0.001
epochs = 10
batch_size = 32

# Define and compile the neural network model
nn_model = Sequential([
    Dense(64, activation='relu', input_shape=(train_data_scaled.shape[1],)),
    Dense(1, activation='sigmoid')  # Output layer with one unit for binary classification
])
optimizer = Adam(learning_rate=learning_rate)
nn_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Iterate through each fold
for fold, (train_index, test_index) in enumerate(skf.split(train_data_scaled, y_train), 1):
    print(f"Fold {fold}:")

    # Get the training and testing data for this fold
    X_train_fold, X_test_fold = train_data_scaled[train_index], train_data_scaled[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]  # Use iloc to access by integer position

    # Fit the model on the training data for this fold
    nn_model.fit(X_train_fold, y_train_fold, epochs=epochs, batch_size=batch_size, verbose=0)

    # Evaluate the model on the training data for this fold
    train_predictions_nn_prob = nn_model.predict(X_train_fold)
    train_predictions_nn = np.round(train_predictions_nn_prob)
    train_accuracy_nn = accuracy_score(y_train_fold, train_predictions_nn)
    print("Training Accuracy (Neural Network):", train_accuracy_nn)
    train_accuracies_nn.append(train_accuracy_nn)

    # Evaluate the model on the testing data for this fold
    test_predictions_nn_prob = nn_model.predict(X_test_fold)
    test_predictions_nn = np.round(test_predictions_nn_prob)
    test_accuracy_nn = accuracy_score(y_test_fold, test_predictions_nn)
    print("Testing Accuracy (Neural Network):", test_accuracy_nn)
    test_accuracies_nn.append(test_accuracy_nn)
    print()  # Print a blank line between folds

# Calculate and print average accuracies across all folds
print("Average Training Accuracy (Neural Network):", np.mean(train_accuracies_nn))
print("Average Testing Accuracy (Neural Network):", np.mean(test_accuracies_nn))


In [ ]:
# Get the weights of the connections between the input layer and the first hidden layer
weights = nn_model.layers[0].get_weights()[0]

# Calculate the absolute mean weights across all folds
abs_mean_weights = np.mean(np.abs(weights), axis=1)

# Get feature names
feature_names = X_train.columns

# Plot the absolute mean weights along with feature names
plt.figure(figsize=(10, 6))
plt.bar(range(len(abs_mean_weights)), abs_mean_weights)
plt.xlabel('Feature Index')
plt.ylabel('Absolute Mean Weight')
plt.xticks(range(len(feature_names)), feature_names, rotation=45)
plt.title('Absolute Mean Weights of Input Layer to First Hidden Layer')
plt.tight_layout()
plt.show()


# ***GridSearch***

In [113]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score

# Initialize StratifiedKFold for cross-validation
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Define the parameter grid for Gradient Boosting
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# Create Gradient Boosting classifier
gb = GradientBoostingClassifier()

# Create GridSearchCV for Gradient Boosting
grid_search_gb = GridSearchCV(estimator=gb, param_grid=param_grid, cv=skf, scoring='accuracy')

# Fit the grid search to the training data
grid_search_gb.fit(X_train, y_train)

# Get the best parameters and best score for Gradient Boosting
best_params_gb = grid_search_gb.best_params_
best_score_gb = grid_search_gb.best_score_

print("Best Parameters for Gradient Boosting:", best_params_gb)
print("Best Score for Gradient Boosting:", best_score_gb)

# Create Gradient Boosting classifier with the best parameters
gb_best = GradientBoostingClassifier(**best_params_gb)

# Fit the model to the entire training data
gb_best.fit(X_train, y_train)

# Evaluate the model on the training data
train_predictions_gb = gb_best.predict(X_train)
train_accuracy_gb = accuracy_score(y_train, train_predictions_gb)
print("Training Accuracy (Gradient Boosting):", train_accuracy_gb)

# Evaluate the model on the testing data
test_predictions_gb = gb_best.predict(X_test)
test_accuracy_gb = accuracy_score(y_test, test_predictions_gb)
print("Testing Accuracy (Gradient Boosting):", test_accuracy_gb)




Best Parameters for Gradient Boosting: {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 200}
Best Score for Gradient Boosting: 0.8978447097865694
Training Accuracy (Gradient Boosting): 0.9047172758512965
Testing Accuracy (Gradient Boosting): 0.897542690545606
time: 5min 24s (started: 2026-11-18 14:50:55 +00:00)


In [ ]:
import matplotlib.pyplot as plt

# Get feature importances and their corresponding names
feature_importances_gb = gb_best.feature_importances_
feature_names = X_train.columns

# Sort features by importance
sorted_indices = feature_importances_gb.argsort()[::-1]
sorted_importances = feature_importances_gb[sorted_indices]
sorted_names = [feature_names[i] for i in sorted_indices]

# Plot the feature importances
plt.figure(figsize=(10, 6))
plt.bar(range(len(sorted_importances)), sorted_importances)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(range(len(sorted_names)), sorted_names, rotation=45)
plt.title('Feature Importance for Gradient Boosting Classifier')
plt.tight_layout()
plt.show()


# ***ENSEMBLE***

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Define models
models = [
    ('Logistic Regression', LogisticRegression(C=10, solver='liblinear', penalty='l2', random_state=42)),
    ('Random Forest', RandomForestClassifier(criterion='gini', max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=50)),
    ('Naive Bayes (Gaussian)', GaussianNB()),
    ('Naive Bayes (Multinomial)', MultinomialNB()),
    ('K-Nearest Neighbors', KNeighborsClassifier(n_neighbors=5, weights='uniform', p=1)),
    ('Decision Tree', DecisionTreeClassifier(criterion='entropy', max_depth=None)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5)),
    ('SVM', SVC(kernel='rbf', C=1)),
    ('Neural Network', MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42))
]

# Create ensemble model using hard voting
ensemble_model = VotingClassifier(estimators=models, voting='hard')

# Fit the ensemble model to the training data
ensemble_model.fit(X_train, y_train)

# Evaluate the ensemble model on the training data
train_predictions_ensemble = ensemble_model.predict(X_train)
train_accuracy_ensemble = accuracy_score(y_train, train_predictions_ensemble)
print("Training Accuracy (Ensemble):", train_accuracy_ensemble)

# Evaluate the ensemble model on the testing data
test_predictions_ensemble = ensemble_model.predict(X_test)
test_accuracy_ensemble = accuracy_score(y_test, test_predictions_ensemble)
print("Testing Accuracy (Ensemble):", test_accuracy_ensemble)

best_model_name = None
best_model_accuracy = 0

# Iterate through each base model in the ensemble
for model_name, model in models:
    # Train the base model
    model.fit(X_train, y_train)

    # Predict with the base model on the training data
    train_predictions_model = model.predict(X_train)

    # Calculate the training accuracy for the base model
    train_accuracy_model = accuracy_score(y_train, train_predictions_model)

    # Predict with the base model on the testing data
    test_predictions_model = model.predict(X_test)

    # Calculate the testing accuracy for the base model
    test_accuracy_model = accuracy_score(y_test, test_predictions_model)

    # Print the accuracy of the base model
    print(f"{model_name} Training Accuracy: {train_accuracy_model}")
    print(f"{model_name} Testing Accuracy: {test_accuracy_model}")

    # Check if this model is the best so far
    if test_accuracy_model > best_model_accuracy:
        best_model_accuracy = test_accuracy_model
        best_model_name = model_name

# Print the best model and its accuracy
print(f"\nBest Model: {best_model_name}")
print(f"Best Model Testing Accuracy: {best_model_accuracy}")


In [ ]:
plt.style.use('dark_background')


In [ ]:
from sklearn.metrics import roc_curve
fper, tper, thresholds = roc_curve(y_test, test_predictions_ensemble)

In [ ]:
plt.plot(fper, tper, color='orange', label='ROC')
plt.plot([0, 1], [0, 1], color='darkblue', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.show()

In [ ]:
!lscpu

In [ ]:
!free -h